In [ ]:
import torch

t_s = torch.tensor([50, 70, 90, 110], dtype=torch.float32)
t_p = torch.tensor([100000, 140000, 180000, 220000], dtype=torch.float32)

t_ns = 0.1 * t_s

def model(t_s, w, b):
    return t_s * w + b

def loss(t_price, t_price_prediction):
    squared_difference = (t_price_prediction - t_price) ** 2
    return squared_difference.mean()

def dloss(t_price, t_price_prediction):
    difference = t_price_prediction - t_price
    return 2 * difference / t_price_prediction.size(0)

def dmodel_dw(t_s):
    return t_s

def dmodel_db():
    return 1.0

def gradient(t_s, t_price, t_price_prediction):
    dloss_p = dloss(t_price, t_price_prediction)
    d_w = dloss_p * dmodel_dw(t_s)
    d_b = dloss_p * dmodel_db()

    return torch.stack([
        d_w.sum(),
        d_b.sum()
    ])

def training_loop(
        n_epochs,
        learning_rate,
        params,
        t_s,
        t_p
):
    for epoch in range(1, n_epochs + 1):
        w, b = params

        # forward pass
        t_price_prediction = model(t_s, w, b)

        # calculate loss
        current_loss = loss(t_p, t_price_prediction)

        # calculate gradients
        grad = gradient(t_s, t_p, t_price_prediction)

        # update parameters
        params = params - learning_rate * grad

        if epoch % 1000 == 0:
            print(
                f"Epoch:  {epoch}, "
                f"Loss: {current_loss.item():2.f}, "
                f"w: {params[0].items():.2f}, "
                f"b: {params[1].items():.2f}"
                )
    return params

# initial parameters
params = torch.tensor(
    [1000.0, 0.0]
)

trained_params = training_loop(
    n_epochs=10000,
    learning_rate=1e-3,
    params=params,
    t_s=t_ns,
    t_p=t_p
)

print("Trained parameters:", trained_params)

# make predictions
w, b = trained_params

pred = model(t_ns, w, b)

print("Predictions:")
print(pred)

print("Actual prices:")
print(t_p)

In [ ]:
import torch

t_s = torch.tensor([1, 2, 3, 4, 5])
t_p = torch.tensor([10, 20, 30, 40, 50])

t_ns = 0.1 * t_s

def model(t_s, w, b):
    return t_s * w + b 

def loss(t_p, t_pp):
    loss_p = (t_p - t_pp) ** 2
    return loss_p.mean()

def dloss(t_p, t_pp):
    dloss_p = (t_p - t_pp) * 2
    return dloss_p / t_p.size(0)

def dloss_w(t_s):
    return t_s

def dloss_b():
    return 1

def gradient(t_s, t_p, t_pp):
    dloss_p = dloss(t_p, t_pp)
    w_dloss = dloss_w(t_s) * dloss_p
    d_dloss = dloss_b() * dloss_p

    return torch.stack([
        w_dloss.sum(),
        d_dloss.sum()
    ])

def training(
        params,
        n_epochs=10000,
        learning_rate=1e-4,
        t_s=t_ns,
        t_p=t_p
):
    w, b = params

    for epoch in range(1, n_epochs + 1):
        t_pp = model(t_s, w, b)

        t_loss = loss(t_p, t_pp)

        t_gradient = gradient(t_s, t_p, t_pp)

        w = w * learning_rate - t_gradient
        b = b * learning_rate - t_gradient

        if (epoch % 1000 == 0):
            print('epoch number: ', epoch)
            print('loss: ', t_loss)

    return w, b

params = [
    torch.tensor([1000]),
    torch.tensor([1])
]

trained_params = training(params, 10000, 1e-4, t_ns, t_p)

pred_values = model(t_s, **trained_params)

print('pred values', pred_values)
print('real values', t_p)
